In [ ]:
import pandas as pd

# Load your data
df = pd.read_csv('/content/train.csv')
print(df)

       sample_id                                    catalog_content  \
0          33127  Item Name: La Victoria Green Taco Sauce Mild, ...   
1         198967  Item Name: Salerno Cookies, The Original Butte...   
2         261251  Item Name: Bear Creek Hearty Soup Bowl, Creamy...   
3          55858  Item Name: Judee’s Blue Cheese Powder 11.25 oz...   
4         292686  Item Name: kedem Sherry Cooking Wine, 12.7 Oun...   
...          ...                                                ...   
74995      41424  Item Name: ICE BREAKERS Spearmint Sugar Free M...   
74996      35537  Item Name: Davidson's Organics, Vanilla Essenc...   
74997     249971  Item Name: Jolly Rancher Hard Candy - Blue Ras...   
74998     188322  Item Name: Nescafe Dolce Gusto Capsules - CARA...   
74999     298504  Item Name: Pimenton de la Vera - Picante (2.47...   

                                              image_link   price  
0      https://m.media-amazon.com/images/I/51mo8htwTH...   4.890  
1      https:

In [ ]:
from sklearn.model_selection import train_test_split

# First, split into training+validation (85%) and test (15%)
train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=42)

# Now split the train_val_df into training (70%) and validation (15%)
train_df, val_df = train_test_split(train_val_df, test_size=0.1765, random_state=42) # 0.1765 * 0.85 = ~0.15

print(f"Total samples: {len(df)}")
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

Total samples: 75000
Training samples: 52498
Validation samples: 11252
Test samples: 11250


In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model

# Define the image input
image_input = Input(shape=(128, 128, 3), name="image_input")

# Load the VGG16 model as the base
vgg_base = VGG16(weights="imagenet", include_top=False, input_tensor=image_input)

# Freeze the layers of the base model
vgg_base.trainable = False

# Flatten the output of the VGG base to a 1D vector
image_features = Flatten(name="flatten")(vgg_base.output)

In [ ]:
from tensorflow.keras.layers import Embedding, Reshape

# Get the number of unique product IDs for the embedding layer
num_unique_ids = df['sample_id'].nunique()

# Define the ID input
id_input = Input(shape=(1,), name="id_input")

# Create the embedding layer
# The output_dim (e.g., 32) is a design choice. It's the size of the vector for each ID.
id_embedding = Embedding(input_dim=num_unique_ids + 1, output_dim=32, name="id_embedding")(id_input)

# Reshape the embedding output to a 1D vector
id_features = Reshape((32,))(id_embedding)

In [ ]:
from tensorflow.keras.layers import concatenate

# Combine the image features and ID features
combined_features = concatenate([image_features, id_features])

# Add the regression head (fully-connected layers)
x = Dense(128, activation="relu")(combined_features)
x = Dense(64, activation="relu")(x)

# The final output layer has one neuron and a linear activation for regression
price_output = Dense(1, activation="linear", name="price_output")(x)

# Create the final model with two inputs and one output
model = Model(inputs=[image_input, id_input], outputs=price_output)

# Print a summary of the model architecture
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 128, 128,  │      1,792 │ image_input[0][0] │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 128, 128,  │     36,928 │ block1_conv1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pool         │ (None, 64, 64,    │          0 │ block1_conv2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv1        │ (None, 64, 64,    │     73,856 │ block1_pool[0][0] │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv2        │ (None, 64, 64,    │    147,584 │ block2_conv1[0][… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 32, 32,    │          0 │ block2_conv2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv1        │ (None, 32, 32,    │    295,168 │ block2_pool[0][0] │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv2        │ (None, 32, 32,    │    590,080 │ block3_conv1[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv3        │ (None, 32, 32,    │    590,080 │ block3_conv2[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_pool         │ (None, 16, 16,    │          0 │ block3_conv3[0][… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv1        │ (None, 16, 16,    │  1,180,160 │ block3_pool[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv2        │ (None, 16, 16,    │  2,359,808 │ block4_conv1[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv3        │ (None, 16, 16,    │  2,359,808 │ block4_conv2[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_pool         │ (None, 8, 8, 512) │          0 │ block4_conv3[0][… │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv1        │ (None, 8, 8, 512) │  2,359,808 │ block4_pool[0][0] │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv2        │ (None, 8, 8, 512) │  2,359,808 │ block5_conv1[0][

 Total params: 18,175,841 (69.34 MB)

 Trainable params: 3,461,153 (13.20 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing import image
from tensorflow.keras.utils import img_to_array
from PIL import Image
import requests
from io import BytesIO
import os
from google.colab import drive

# --- 1. Compile the Model ---
print("Compiling the model...")
model.compile(
    optimizer='adam',
    loss='mean_absolute_error',
    metrics=['mean_absolute_error']
)

# --- 2. Create a Custom Data Generator ---
class ProductDataGenerator(Sequence):
    def __init__(self, df, id_to_index, batch_size=8, target_size=(128, 128), shuffle=True):
        self.df = df.reset_index(drop=True)
        self.id_to_index = id_to_index # Use the pre-computed mapping
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(df))
        self.on_epoch_end()

    def __len__(self):
        # number of batches per epoch
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        # Generate indexes of the batch
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indexes]
        return self.__data_generation(batch_df)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_df):
      X_images = []
      X_ids = []
      y_prices = []

      for _, row in batch_df.iterrows():
          try:
              response = requests.get(row['image_link'], timeout=5)
              img = Image.open(BytesIO(response.content)).convert("RGB")
              img = img.resize(self.target_size)
              img_array = img_to_array(img) / 255.0
          except Exception:
              # Use a placeholder for failed image loads
              img_array = np.zeros((*self.target_size, 3))

          X_images.append(img_array)
          # Use the mapped index for the sample_id
          X_ids.append(self.id_to_index[row["sample_id"]])
          y_prices.append(row["price"])

      X_images = np.array(X_images, dtype=np.float32)
      X_ids = np.array(X_ids).reshape(-1, 1)
      y_prices = np.array(y_prices).reshape(-1, 1)

      # Return tuple instead of list
      return (X_images, X_ids), y_prices


# --- 3. Create Generators for Training and Validation ---
batch_size = 32  # you can tune this based on your GPU/CPU memory

# Create a mapping from sample_id to a contiguous integer range using the full dataframe
unique_ids = df['sample_id'].unique()
id_to_index = {id: i for i, id in enumerate(unique_ids)}

train_generator = ProductDataGenerator(train_df, id_to_index, batch_size=batch_size, shuffle=True)
val_generator = ProductDataGenerator(val_df, id_to_index, batch_size=batch_size, shuffle=False)

# --- 4. Train the Model ---
print("\nStarting model training...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=2,
    verbose=1
)

# --- 5. Save the Trained Model ---
print("\nTraining complete. Saving model...")
# Mount Google Drive
drive.mount('/content/drive')
# Define the path to save the model in Google Drive
model_save_path = '/content/drive/My Drive/product_price_predictor.h5'
model.save(model_save_path)
print(f"✅ Model saved successfully to {model_save_path}")

In [ ]:
from tensorflow.keras.models import load_model
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to load the model from Google Drive
model_load_path = '/content/drive/My Drive/product_price_predictor.h5'

# Load the model
if os.path.exists(model_load_path):
    model = load_model(model_load_path)
    print("✅ Model loaded successfully!")
else:
    print(f"❌ Model file not found at {model_load_path}. Please ensure the training cell was run and the model was saved to Google Drive.")

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("product_price_predictor.h5")
print("✅ Model loaded successfully!")
